# 06 — Sensitivity to component count K

This notebook shows what under-fitting and over-fitting look like by reconstructing a known two-component profile with K below, equal to, and above the truth. It uses the real forward model and the reproduced sigma-only Adam fit, holding amplitudes and centroids at peak-init values and optimising sigmas.

Modules exercised:

- `tools.data.gaussians.GaussianMixture` (real, imported)
- sigma-only Adam scan reproduced from `pipelines.param_pipeline.sigma.SigmaScan`

Fixed seed, CPU only.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

SEED = 0
rng  = np.random.default_rng(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
except Exception:
    torch = None

plt.rcParams.update({
    "figure.dpi"     : 110,
    "savefig.dpi"    : 110,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.30,
    "grid.linewidth" : 0.4,
})

print("Repo root :", REPO_ROOT)
print("NumPy     :", np.__version__)


In [ ]:
def gaussian_mixture_profile(height_axis, components, noise_std=0.0, rng=None):
    profile = np.zeros_like(height_axis, dtype=np.float64)
    for amp, mu, sigma in components:
        profile += amp * np.exp(-((height_axis - mu) ** 2) / (2.0 * sigma * sigma))
    if noise_std > 0.0:
        draw    = rng.normal(0.0, noise_std, size=height_axis.shape) if rng is not None else np.random.normal(0.0, noise_std, size=height_axis.shape)
        profile = profile + draw
    return profile.astype(np.float32)


def pack_parameters(components_per_pixel, k_max, shape):
    Az, R  = shape
    params = np.zeros((3 * k_max, Az, R), dtype=np.float32)
    for (az, rg), comps in components_per_pixel.items():
        for k, (amp, mu, sigma) in enumerate(comps[:k_max]):
            params[3 * k    , az, rg] = amp
            params[3 * k + 1, az, rg] = mu
            params[3 * k + 2, az, rg] = sigma
    return params


In [ ]:
from tools.data.gaussians import GaussianMixture

def per_pixel_pred(sigmas, height_axis, amps, mus):
    safe_s2 = 2.0 * np.maximum(sigmas, 1e-6) ** 2
    diff    = height_axis[None, :] - mus[:, None]
    expon   = np.clip(-(diff ** 2) / safe_s2[:, None], -100.0, 0.0)
    return (amps[:, None] * np.exp(expon)).sum(axis=0)

def per_pixel_loss(sigmas, height_axis, profile, amps, mus):
    return float(np.mean((per_pixel_pred(sigmas, height_axis, amps, mus) - profile) ** 2))

def loss_grad(sigmas, height_axis, profile, amps, mus, eps=1e-5):
    g = np.zeros_like(sigmas)
    for k in range(sigmas.size):
        sp = sigmas.copy(); sp[k] += eps
        sm = sigmas.copy(); sm[k] -= eps
        g[k] = (per_pixel_loss(sp, height_axis, profile, amps, mus) - per_pixel_loss(sm, height_axis, profile, amps, mus)) / (2 * eps)
    return g

def adam_scan(s0, height_axis, profile, amps, mus, lo, hi, n_steps=400, lr=0.2, b1=0.95, b2=0.999):
    eps = 1e-8
    s = np.clip(s0.astype(np.float64), lo, hi); m = np.zeros_like(s); v = np.zeros_like(s)
    for t in range(n_steps):
        g  = loss_grad(s, height_axis, profile, amps, mus)
        m  = b1 * m + (1 - b1) * g
        v  = b2 * v + (1 - b2) * g * g
        tf = t + 1.0
        s  = np.clip(s - lr * (m / (1 - b1 ** tf)) / (np.sqrt(v / (1 - b2 ** tf)) + eps), lo, hi)
    return s

print('helpers ready')

## Two-component truth, three values of K

We seed amplitudes and centroids at the K strongest smoothed maxima (a simplified peak-init), fit the sigmas, and reconstruct. K = 1 cannot represent both lobes; K = 2 matches the truth; K = 3 adds a redundant component.

In [ ]:
H           = 220
height_axis = np.linspace(0.0, 45.0, H).astype(np.float64)
dh          = float(height_axis[1] - height_axis[0]); h_span = float(height_axis[-1] - height_axis[0])
lo, hi      = dh, h_span / 2.0

true_comps = [(1.0, 14.0, 2.5), (0.7, 30.0, 3.0)]
profile    = gaussian_mixture_profile(height_axis, true_comps, noise_std=0.01, rng=rng).astype(np.float64)

from scipy.ndimage import uniform_filter1d
from scipy.signal  import find_peaks
smoothed     = uniform_filter1d(profile, size=5, mode='nearest')
peaks, props = find_peaks(smoothed, prominence=smoothed.max() * 0.05)
order        = peaks[np.argsort(props['prominences'])[::-1]]

def seed_and_fit(K):
    if order.size >= K:
        idxs = order[:K]
    else:
        idxs = np.concatenate([order, np.linspace(0, H - 1, K - order.size, dtype=int)])
    amps = np.array([float(smoothed[i]) for i in idxs[:K]])
    mus  = np.array([float(height_axis[i]) for i in idxs[:K]])
    s0   = np.full(K, max(2 * dh, h_span / (8.0 * K)))
    sf   = adam_scan(s0, height_axis, profile, amps, mus, lo, hi)
    pred = per_pixel_pred(sf, height_axis, amps, mus)
    ss_res = float(np.sum((pred - profile) ** 2)); ss_tot = float(np.sum((profile - profile.mean()) ** 2)) + 1e-12
    return pred, 1.0 - ss_res / ss_tot

fits = {K: seed_and_fit(K) for K in (1, 2, 3)}
for K, (pred, r2) in fits.items():
    print(f'K={K}  R2={r2:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, K in zip(axes, (1, 2, 3)):
    pred, r2 = fits[K]
    ax.plot(height_axis, profile, color='black', lw=1.4, label='truth')
    ax.plot(height_axis, pred, color='tab:red', lw=1.3, ls='--', label='K-component fit')
    ax.set_title(f'K = {K}   R2 = {r2:.3f}')
    ax.set_xlabel('height h [m]')
axes[0].set_ylabel('intensity')
axes[0].legend(fontsize=8)
fig.suptitle('Reconstruction quality vs component count K (truth has 2 components)')
fig.tight_layout()
plt.show()

## Expected outcome

K = 1 should visibly under-fit, missing one lobe and giving a markedly lower R2. K = 2 should overlay the truth with high R2. K = 3 should also fit well but with no meaningful R2 gain over K = 2, illustrating diminishing returns that the best-K penalty in notebook 05 is designed to discourage.